In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import os, shutil, multiprocessing
from PIL import Image

import torch
import torch.nn as nn
from torchvision import transforms, datasets
from torch.utils.data import TensorDataset, DataLoader, random_split
import torch.optim as optim


In [3]:
device = ('cuda' if torch.cuda.is_available() else 'cpu')
print(device)

cuda


In [4]:
transform_train = transforms.Compose([
    transforms.Resize(120),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.RandomCrop(112),
    transforms.ColorJitter(brightness=0.2, contrast=0.2),
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
])

transform_test = transforms.Compose([
    transforms.Resize(120),
    transforms.CenterCrop(112),
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
])

In [5]:
src_folder = r"C:\Users\Tanvir\Desktop\ML Minor Project\Dogs vs Cats Classifier\data\train"
dst_folder = r"C:\Users\Tanvir\Desktop\ML Minor Project\Dogs vs Cats Classifier\data\train1"

for fname in os.listdir(src_folder):
    if fname.startswith('cat'):
        dest = os.path.join(dst_folder, 'cat')
    elif fname.startswith('dog'):
        dest = os.path.join(dst_folder, 'dog')
    else:
        continue

    os.makedirs(dest, exist_ok=True)
    shutil.copy(os.path.join(src_folder, fname), dest)

print('Done')

Done


In [6]:
dataset = datasets.ImageFolder(r"C:\Users\Tanvir\Desktop\ML Minor Project\Dogs vs Cats Classifier\data\train1", transform=transform_train)

In [7]:
train_size = int(0.8 * len(dataset))
val_size = int(len(dataset) - train_size)

In [8]:
train_set, val_set = random_split(dataset, [train_size, val_size])

In [9]:
train_loader = DataLoader(train_set, batch_size=64, shuffle=True, num_workers=8)
val_loader = DataLoader(val_set, batch_size=64, num_workers=8)

In [10]:
class CNN(nn.Module):
    def __init__(self):
        super().__init__()

        self.conv_lyr = nn.Sequential(
            nn.Conv2d(3, 64, 3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.Conv2d(64, 64, 3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d(2, 2),
            nn.Dropout2d(0.2),

            nn.Conv2d(64, 128, 3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(),
            nn.Conv2d(128, 128, 3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(),
            nn.MaxPool2d(2,2),
            nn.Dropout2d(0.3),

            nn.Conv2d(128, 256, 3, padding=1),
            nn.BatchNorm2d(256),
            nn.ReLU(),
            nn.Conv2d(256, 256, 3, padding=1),
            nn.BatchNorm2d(256),
            nn.ReLU(),
            nn.MaxPool2d(2,2),
            nn.Dropout2d(0.4),
        )

        self.linear_lyr = nn.Sequential(
            nn.Flatten(),

            nn.LazyLinear(512),
            nn.BatchNorm1d(512),
            nn.ReLU(),
            nn.Dropout(0.4),

            nn.Linear(512, 1),
        )
    def forward(self, x):
        x = self.conv_lyr(x)
        x = self.linear_lyr(x)
        return x

In [13]:
model = CNN().to(device)
optimizer = optim.Adam(model.parameters())
critrion = nn.BCEWithLogitsLoss()

In [ ]:
epochs = 25
best_accuracy = 0.0

for epoch in range(epochs):
    model.train()
    running_loss = 0.0
    total = 0
    correct = 0
    for imgs, labels in train_loader:
        imgs = imgs.to(device)
        labels = labels.float().unsqueeze(1).to(device)

        optimizer.zero_grad()
        output = model(imgs)
        loss = critrion(output, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item()
    total_loss = running_loss/len(train_loader)
    
    model.eval()
    with torch.no_grad():
        for imgs, labels in val_loader:
            imgs = imgs.to(device)
            labels = labels.float().unsqueeze(1).to(device)
            output = model(imgs)

            pred = torch.sigmoid(output) > 0.5
            total += labels.size(0)
            correct += (labels == pred).sum().item()

        val_acc = correct/total*100
        if val_acc > best_accuracy:
            best_accuracy = val_acc
            torch.save(model.state_dict(), 'bset_model.pth')
            
    print(f'loss for Epoch-{epoch+1} is = {total_loss:.4f}, accuracy is {val_acc:.2f}')




loss for Epoch-16 is = 0.2200, accuracy is 91.34
loss for Epoch-17 is = 0.2203, accuracy is 91.62
loss for Epoch-18 is = 0.2110, accuracy is 91.69
loss for Epoch-19 is = 0.2077, accuracy is 91.84
loss for Epoch-20 is = 0.2033, accuracy is 91.96
loss for Epoch-21 is = 0.1943, accuracy is 92.14
loss for Epoch-22 is = 0.1858, accuracy is 92.33
loss for Epoch-23 is = 0.1802, accuracy is 92.50
loss for Epoch-24 is = 0.1828, accuracy is 92.63
loss for Epoch-25 is = 0.1743, accuracy is 92.75
